# Motif Sample Notebook

Put the motif_sample_notebook folder in the main directory of the Motif repository.

Install the Conda environment from the main Motif directory:

```bash
conda env create -f environment.yml
```

Open the notebook in Jupyter from the motif_sample_notebook folder and select motif Conda environment. Run all code cells after another. They execute all preparation scripts and pipeline stages A to G. All generated files are saved in motif_sample_notebook.

The THYM sample contains 121 samples, 200 genes and 500 CpGs. This example uses gene–CpG pairs that were significant in an earlier analysis. It just shows how to run the pipeline.

## 1. Set up

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

assert Path.cwd().name == "motif_sample_notebook", "Run this notebook from motif_sample_notebook/."

os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = str(Path.cwd().parent)
os.environ["SNAKEMAKE_PROFILE"] = str(Path("sample_snakemake.yaml").resolve())

## 2. Prepare the cohorts data

Split the raw inputs, convert log-counts to integers and build metadata.

In [2]:
%%bash
set -e
python -m pipeline.split_data_to_cohorts \
    --metadata data/raw/GDC-PANCAN.basic_phenotype.tsv \
    --expression data/raw/GDC-PANCAN.htseq_fpkm-uq.tsv \
    --methylation data/raw/GDC-PANCAN.methylation450.tsv \
    --raw-counts data/raw/GDC-PANCAN.htseq_counts.tsv \
    --output-dir splitted_by_project
python -m scripts.fill_cohort_dirs --config config.yaml --split-dir splitted_by_project --data-dir data
python -m scripts.repair_raw_counts data/THYM/raw_counts.tsv
python -m scripts.build_metadata --metadata data/raw/GDC-PANCAN.basic_phenotype.tsv --config config.yaml --data-directory data

## 3. Prepare references

Prepare gene and CpG annotations and the genome file for the pipeline.


In [3]:
%%bash
set -e
python -m scripts.build_gene_annotation \
    --gencode-file resources/gencode.v22.annotation.gtf.gz \
    --hgnc-file resources/hgnc_complete_set.txt \
    --gene-annotation-output resources/gene_annotation.v22.tsv \
    --tss resources/gene_annot.tsv \
    --coding-genes resources/gencode.v22.protein_coding_gene_ids.txt
python -m scripts.build_cpg_coordinates \
    --cpg-map resources/HM450.hg38.manifest.gencode.v36.probeMap --output-file resources/cpg_annot.tsv
python -m scripts.build_cpg_annotations_for_f1 \
    --cpg-annotations-file resources/HM450.hg38.manifest.gencode.v36.zhou.tsv.gz \
    --gencode-file resources/gencode.v22.annotation.gtf.gz --output resources/cpg_gene_annotations.hg38.gencode.v22.tsv.gz
gzip -dc resources/hg38.fa.gz > resources/hg38.fa
python -m scripts.build_fasta_index resources/hg38.fa

## 4. Run Motif pipeline

In [4]:
%%bash
set -e
python -m pipeline.run_pipeline all --executor local --cores 2

Using profile /Users/juliamarlene/Downloads/Motif/motif_sample_notebook/sample_snakemake.yaml for setting default command line arguments.
Assuming unrestricted shared filesystem usage.
host: MacBook-Air-von-Julia.local
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 2
Rules claiming more threads will be scaled down.
Provided resources: prepare_jobs=Resource("prepare_jobs", 1), grn_jobs=Resource("grn_jobs", 1)
Job stats:
job                          count
-------------------------  -------
prepare_data                     1
grn_boost_runs                   4
sex_filter                       1
reverse_causality_filter         1
stability                        1
stability_comparison             1
stability_selection              1
aggregate_runs                   1
select_top_mfs                   1
preprocess_evaluation            1
gene_set_enrichment              1
mediation                        1
ppi_proximity                    1
prepare_chipseq_resources        1
b

## 5. View results

In [5]:
results = Path("results/selected_settings/rank_1/THYM")
evaluation = results / "evaluation"
pd.reset_option("display.float_format")

print("Selected MFs")
mfs = pd.read_csv(results / "selected_mfs/mfs.tsv", sep="\t")
display(mfs[["rank", "MF_ensembl", "mfs_scores"]])

print("F1 — Binding sites (unadjusted p-values)")
binding = pd.read_csv(evaluation / "binding_sites.tsv", sep="\t")
display(binding[["track", "n_mf", "p_value_sign_test"]])

print("F2 — PPI proximity (unadjusted p-values)")
proximity = pd.read_csv(evaluation / "ppi_proximity.tsv", sep="\t")
display(proximity[["measure", "n_mfs_in_network", "p_value"]])

print("F3 — Gene-set enrichment (BH-adjusted p-values)")
enrichment = pd.read_csv(evaluation / "gene_set_enrichment.tsv", sep="\t")
reported_terms = enrichment.loc[enrichment["term_has_min_mf_overlap"]]
display(reported_terms.sort_values(["adj_p_value", "p_value"])[["library_name", "term_name", "n_mf_symbols_in_term", "p_value", "adj_p_value"]])

print("F4 — Mediation (BY-adjusted joint p-values)")
triplets = pd.read_csv(evaluation / "mediation_triplets.tsv", sep="\t")
display(triplets.sort_values("joint_adj_p_value")[["MF", "CpG", "G_prime", "a_pvalue", "b_pvalue", "joint_adj_p_value"]])

Selected MFs


,rank,MF_ensembl,mfs_scores
0,1,ENSG00000164161,51.90600
1,2,ENSG00000122122,39.59050
2,3,ENSG00000100351,31.39650
3,4,ENSG00000182866,30.20525
4,5,ENSG00000118513,24.74125
5,6,ENSG00000158488,22.60875
6,7,ENSG00000004468,20.59100
7,8,ENSG00000278195,16.34300
8,9,ENSG00000100060,12.96000
9,10,ENSG00000100027,12.62200


F1 — Binding sites (unadjusted p-values)


,track,n_mf,p_value_sign_test
0,FIMO,1,1.0
1,ChIP_seq,0,NaN


F2 — PPI proximity (unadjusted p-values)


,measure,n_mfs_in_network,p_value
0,pagerank,10,0.034826
1,harmonic,10,0.358209


F3 — Gene-set enrichment (BH-adjusted p-values)


,library_name,term_name,n_mf_symbols_in_term,p_value,adj_p_value
48,Reactome_2022,Adaptive Immune System R-HSA-1280218,3,0.000091,0.172133
434,Reactome_2022,Signal Transduction R-HSA-162582,6,0.001696,0.387300
223,Reactome_2022,Immune System R-HSA-168256,3,0.024628,0.791597


F4 — Mediation (BY-adjusted joint p-values)


,MF,CpG,G_prime,a_pvalue,b_pvalue,joint_adj_p_value
539,SASH3,cg15728692,PLXNB2,2.115519e-31,3.309576e-29,1.591299e-24
4572,SSTR3,cg15728692,PLXNB2,9.984423e-30,4.907681e-29,1.591299e-24
681,SASH3,cg18982286,PLXNB2,1.995293e-32,8.135265e-29,1.758554e-24
1003,MFNG,cg18982286,PLXNB2,2.631658e-31,5.468639e-28,8.865937e-24
407,MFNG,cg15728692,PLXNB2,3.431926e-28,8.176055e-27,1.060423e-22
...,...,...,...,...,...,...
3253,GRAP2,cg06057569,PANX2,2.279811e-28,1.175256e-01,1.000000e+00
3252,GRAP2,cg06057569,MLC1,2.279811e-28,3.511419e-01,1.000000e+00
3247,LCK,cg17237086,CACNA1I,3.111941e-16,9.077518e-02,1.000000e+00
3257,GRAP2,cg06057569,SBF1,2.279811e-28,7.503454e-01,1.000000e+00
